<img src="https://hilpisch.com/tpq_logo_bic.png"
width="20%" align="right">

<a href="https://colab.research.google.com/github/yhilpisch/algocolab/blob/main/notebooks/00_colab_introduction.ipynb"
target="_blank"><img
src="https://colab.research.google.com/assets/colab-badge.svg"
alt="Open In Colab"/></a>


# Algorithmic Trading with Python & Google Colab

## Starter — Google Colab as a Quantitative Lab

### Drive, Packages, GPUs, and AI-Assisted Debugging

&copy; Dr. Yves J. Hilpisch  
The Python Quants GmbH | https://tpq.io  
https://hilpisch.com

This short companion notebook demonstrates why Colab is useful for the
webinar: Drive persistence, a ready scientific Python stack, optional GPU
acceleration, and an integrated Gemini debugging workflow.

Colab runtimes are temporary. Treat `/content` as scratch space and persist
anything important to Google Drive.


## 1. Connect the Runtime to Google Drive

Drive lets separate Colab runtimes share data, notebooks, checkpoints, and
experiment artifacts. The mount cell is intended for Colab; the local fallback
keeps this notebook inspectable outside Colab.


In [ ]:
from pathlib import Path
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    drive.mount('/content/drive')  # connect persistent Google Drive storage
    RUNS_ROOT = Path('/content/drive/MyDrive/algo/runs')
else:
    RUNS_ROOT = Path('/Users/yves/Google Drive/My Drive/algo/runs')
RUNS_ROOT.mkdir(parents=True, exist_ok=True)  # create the shared artifact root
print(f'Runtime: {"Colab" if IN_COLAB else "local"}')
print(f'Persistent artifacts: {RUNS_ROOT}')

## 2. Use the Scientific Python Stack Immediately

Colab commonly provides `numpy`, `pandas`, `matplotlib`, `scipy`, `statsmodels`,
and `torch`. The exact versions can change, so record them in a serious run.


In [ ]:
import importlib
packages = ['numpy', 'pandas', 'matplotlib', 'scipy', 'statsmodels', 'torch']
for name in packages:
    module = importlib.import_module(name)  # import the available package
    print(f'{name}: {getattr(module, "__version__", "unknown")}')

## 3. Compare CPU and GPU Matrix Multiplication

Matrix multiplication is a deliberately simple proxy for tensor workloads.
The benchmark measures elapsed time on the available devices; it is not a
promise about end-to-end trading speed. Small matrices can be slower on a GPU
because transfer and startup overhead dominate.


In [ ]:
import time
import torch
size = 3 * 2048  # large enough to make GPU parallelism visible
device_names = ['cpu']
if torch.cuda.is_available():
    device_names.append('cuda')
for device_name in device_names:
    device = torch.device(device_name)
    left = torch.randn((size, size), device=device)  # allocate on the device
    right = torch.randn((size, size), device=device)
    if device.type == 'cuda':
        torch.cuda.synchronize()  # wait before starting the timed region
    started = time.perf_counter()
    product = left @ right  # run the matrix multiplication
    if device.type == 'cuda':
        torch.cuda.synchronize()  # include asynchronous GPU work
    elapsed = time.perf_counter() - started
    print(f'{device_name}: {elapsed:.3f} seconds')

## 4. A Financial Bug: Misaligning Signal and Return

A common backtest error uses today's return with today's signal. That gives
the strategy information from the period it is supposedly predicting. The
correct convention is: build the signal using information through time `t`,
then apply it to the return observed at `t+1`.


In [ ]:
import numpy as np
import pandas as pd
returns = pd.Series([0.01, -0.02, 0.03], name='return')
signal = np.sign(returns)  # uses today's return: forbidden information
wrong_pnl = signal * returns  # look-ahead-biased backtest result
correct_signal = signal.shift(1).fillna(0)  # trade after the signal exists
correct_pnl = correct_signal * returns  # apply yesterday's signal today
pd.DataFrame({'return': returns, 'signal': signal,
              'wrong_pnl': wrong_pnl,
              'correct_signal': correct_signal,
              'correct_pnl': correct_pnl})

### Ask Gemini to Diagnose the Bug

In Colab, select the suspicious cell and ask Gemini to explain the timing
convention. A useful prompt is:

> Audit this backtest for look-ahead bias. State the timestamp at which each
> signal is known, identify whether the position is shifted before applying the
> return, and rewrite the smallest correct code change. Do not assume that a
> profitable result is valid evidence.

Gemini can accelerate debugging, but the learner still verifies the timestamps,
the shifted arrays, and the economic interpretation.


## 5. Colab Niceties and Boundaries

- Use `%%time` or `time.perf_counter()` for quick measurements.
- Keep exploratory plots and tables close to the code that creates them.
- Save checkpoints and run manifests to Drive, not only `/content`.
- Pin or record package versions when results matter.
- Restart and rerun from the top before calling a notebook reproducible.
- GPU availability, session lifetime, and package versions can vary.

Colab is a convenient research lab—not a guarantee of persistent hardware or
production-grade execution.


---

<img src="https://hilpisch.com/tpq_logo_bic.png"
width="20%" align="right">
